# Detyra për Shtëpi 6 — Komponentet e Lidhura Fort & Renditjet Topologjike

---

| Tema | Ushtrimi i laboratorit |
|------|------------------------|
| **Komponentet e Lidhura Fort (SCC)** | Ushtrimi 4 — `bfs_reach`, `analyze_scc` |
| **Renditjet Topologjike** | Ushtrimi 5/6 — `build_predecessors`, `available_nodes` |

---

> **Udhëzim i përgjithshëm**
> Çdo problem ka dy pjesë:
> **A — Puna me shkrim:** analizoni grafin me letër, plotësoni tabelat.
> **B — Kodi:** plotësoni boshllëqet `# PLOTESO` duke thirrur funksionet ekzistuese
> nga ushtrimi përkatës i laboratorit — nuk shkruani asnjë funksion të ri.

---

## Problemi 1 — Komponentet e Lidhura Fort (SCC)

### Kujtues teorik — Ushtrimi 4 i laboratorit

Në Ushtrimin 4 ndërtuat një graf të drejtuar interaktivisht dhe zgjidhët grupe kulmesh
për të kontrolluar nëse formojnë **Komponent të Lidhur Fort (SCC)**.

Logjika qendrore ishte funksioni `analyze_scc(adj, selected)`:

1. Ekzekuton **BFS nga çdo kulm i zgjedhur** → merr `reach[v]` (kulmet e arritshme nga v).
2. Kërkon çifte **(u, v)** ku nuk ekziston shteg i drejtuar u → v (`missing`).
3. Nëse `missing` është bosh → bashkësia e zgjedhur **është SCC**; ndryshe **nuk është**.

> **Vetia themelore:** u dhe v janë në të njëjtin SCC nëse dhe vetëm nëse
> `v ∈ reach[u]`  **DHE**  `u ∈ reach[v]`.

### A — Puna me shkrim

Jepet grafi i drejtuar **G₅** me 5 kulme:

```
Kulmet  : 0, 1, 2, 3, 4
Brinjët: 0→1,  1→2,  2→0,  2→3,  3→4,  4→3
```

```
  +---<---<---+
  |           |
 [0] -->[1]-->[2]-->[3]<--+
                      |   |
                     [4]--+
```

*Cikli i parë: 0→1→2→0. Cikli i dytë: 3→4→3. Brinja njëkahore: 2→3.*

---

#### A1 — Bashkësia e arritshme (reach)

Gjurmoni BFS dhe plotësoni tabelën vetëm për kulmet 0 dhe 3
(mjaftojnë për të kuptuar strukturën e dy cikleve).

| Kulmi src | reach[src] — të gjithë kulmet e arritshme (përfshi src) |
|:---------:|----------------------------------------------------------|
| 0         |                                                          |
| 3         |                                                          |

---

#### A2 — Analiza e SCC-ve

Duke përdorur logjikën e `analyze_scc` nga laboratori,
plotësoni tabelën duke vendosur **PO / JO** dhe arsyen:

| Bashkësia e zgjedhur | SCC? | Arsyeja (çifti **(u,v)** që mungon, ose konfirmimi) |
|----------------------|------|-----------------------------------------------------|
| {0, 1, 2}            |      |                                                     |
| {3, 4}               |      |                                                     |
| {0, 1, 2, 3}         |      |                                                     |
| {2, 3}               |      |                                                     |

---

#### A3 — Pyetje konceptuale

Brinja **2→3** lidh SCC-në {0,1,2} me SCC-në {3,4}.
Pse kjo brinjë nuk mund të jetë kurrë e dyanshme (pra nuk mund të ketë edhe 3→2)?
Çfarë do të ndodhte nëse do të ekzistonte brinja 3→2?

*Shkruani përgjigjen tuaj këtu.*

In [ ]:
# ================================================================
# Problemi 1 — Komponentet e Lidhura Fort
# Funksionet bfs_reach dhe analyze_scc jane identike me Ushtrimin 4.
# Plotesoni vetem boshlleket e shenuar me  # PLOTESO
# ================================================================

from collections import deque

# --- Grafi G5 ---
adj_G5 = {
    0: [1],
    1: [2],
    2: [0, 3],
    3: [4],
    4: [3],
}
n_G5 = 5


# --- Funksionet nga Ushtrimi 4 — mos i ndrysho ---

def bfs_reach(adj, src):
    """BFS nga src; kthen bashkesine e kulmeve te arritshme."""
    visited = {src}
    q = deque([src])
    while q:
        v = q.popleft()
        for w in adj.get(v, []):
            if w not in visited:
                visited.add(w)
                q.append(w)
    return visited


def analyze_scc(adj, selected):
    """
    Kontrollon nese kulmet ne 'selected' formojne nje SCC.
    Kthen True nese po, False nese jo.
    (Logjika identike me analyze_scc te Ushtrimit 4.)
    """
    sel = sorted(selected)
    if len(sel) <= 1:
        return True                         # nje kulm i vetem

    reach = {v: bfs_reach(adj, v) for v in sel}
    missing = [(u, v) for u in sel for v in sel
               if u != v and v not in reach[u]]

    if missing:
        print(f"  JO SCC — shtigjet qe mungojne: {missing}")
    else:
        print(f"  PO — SCC i konfirmuar!")
    return len(missing) == 0


# --- Pjesa B: verifikoni pergjigjet e punes me shkrim ---

print("=== Verifikimi i bashkesive ===")

print("\n{0, 1, 2}:")
analyze_scc(adj_G5, {0, 1, 2})

print("\n{3, 4}:")
analyze_scc(adj_G5, {3, 4})

print("\n{0, 1, 2, 3}:")
analyze_scc(adj_G5, {0, 1, 2, 3})

print("\n{2, 3}:")
analyze_scc(adj_G5, {2, 3})


# ----------------------------------------------------------------
# Shtesë e vogël: gjej të gjithë kulmet ne të njëjtin SCC me v.
# Përdorni bfs_reach dy herë: një herë në grafit normal,
# dhe një herë duke kontrolluar nëse v arrihet nga w.
# ----------------------------------------------------------------

def gjej_scc_te_kulmit(adj, n, v):
    """
    Gjen bashkesine e kulmeve qe i perkasin te njejtin SCC me kulmin v.
    Perdor bfs_reach (nga Ushtrimi 4) — nuk shkruan logjike te re.
    """
    arritshme_nga_v = bfs_reach(___, ___)    # PLOTESO: bfs_reach nga kulmi v

    scc = set()
    for w in range(n):
        if w in arritshme_nga_v:            # w arrihet nga v
            arritshme_nga_w = bfs_reach(adj, w)
            if ___ in arritshme_nga_w:      # PLOTESO: v arrihet nga w?
                scc.add(w)
    return scc


print("\n=== SCC-ja e kulmit 0 ===")
print(gjej_scc_te_kulmit(adj_G5, n_G5, 0))   # pritet {0, 1, 2}

print("\n=== SCC-ja e kulmit 3 ===")
print(gjej_scc_te_kulmit(adj_G5, n_G5, 3))   # pritet {3, 4}

---

## Problemi 2 — Renditjet Topologjike

### Kujtues teorik — Ushtrimi 5 dhe 6 i laboratorit

Në Ushtrimin 5 klikuat lendet universitare **një nga një** sipas rendit topologjik.
Sistemi verifikonte nëse kishit plotësuar të gjitha parakushtet para se të zgjidhni.

Logjika qendrore ishte funksioni `available_nodes(pred, placed, n)`:

> Kthen listën e kulmeve **të gatshme për t'u vendosur** — ato ku
> të gjithë paraardhësit (`pred[v]`) janë tashmë në `placed`.

Ushtrimi 6 e shtriu këtë logjikë me grafin tuaj dhe shtoi `has_cycle` për të
kontrolluar nëse renditja topologjike ishte e mundur fare.

> **Kujtesë:** renditja topologjike ekziston **vetëm për DAG** (graf pa cikle).

### A — Puna me shkrim

Jepet DAG-u i hapave të vendosjes së një aplikacioni:

| Nyjja | Emri                  | Parakushtet (pred) |
|:-----:|------------------------|---------------------|
| 0     | Instalo Python         | — (asnjë)           |
| 1     | Instalo pip            | 0 (Python)          |
| 2     | Instalo bibliotekat    | 0 (Python), 1 (pip) |
| 3     | Konfiguro `.env`       | — (asnjë)           |
| 4     | Ekzekuto aplikacionin  | 2 (bibliotekat), 3 (.env) |

**Brinjët:** (0,1), (0,2), (1,2), (2,4), (3,4)

```
  [0=Python] ---> [1=pip] ---> [2=bibliotekat]
       |                            |
       +--------------------------->+          [3=.env]
                                    |             |
                                    +---------->[4=app]
```

---

#### B1 — Gjurmimi i `available_nodes` hap pas hapi

Plotësoni tabelën. Në çdo hap zgjidhni **çfarëdo kulmi** nga lista e disponueshme
(ka shumë renditje të vlefshme — zgjedhja juaj mund të ndryshojë nga e shokut).

`available_nodes` kthen kulmet ku **të gjithë** pred[v] janë në `placed`.

| Hapi | `placed` deri tani | `available_nodes(...)` | Zgjidhni | `rendi` deri tani |
|:----:|--------------------|------------------------|----------|-------------------|
| 1    | { }                |                        |          |                   |
| 2    |                    |                        |          |                   |
| 3    |                    |                        |          |                   |
| 4    |                    |                        |          |                   |
| 5    |                    |                        |          |                   |

**Renditja juaj topologjike:** ___________________________

---

#### B2 — Shumësia e renditjeve

Në hapin 1, `available_nodes` kthen dy kulme të mundshme.
Shkruani **të gjitha renditjet topologjike të vlefshme** që mund të gjeneroni
duke ndryshuar zgjedhjen në hapin e parë (dhe hapin e dytë nëse ka variante):

1. ____________________
2. ____________________
3. ____________________  *(nëse ka)*

---

#### B3 — Pyetje konceptuale

Shtoni brinjën **(4→0)** në DAG (aplikacioni → Python — e pamundur realisht, por
supozojeni për qëllim teorik). Çfarë ndodh me renditjen topologjike?
Çfarë do të kthente `has_cycle` nga Ushtrimi 6?

*Shkruani përgjigjen tuaj këtu.*

In [ ]:
# ================================================================
# Problemi 2 — Renditjet Topologjike
# Funksionet build_predecessors dhe available_nodes jane identike
# me Ushtrimin 5 dhe 6 te laboratorit.
# Plotesoni vetem boshlleket e shenuar me  # PLOTESO
# ================================================================

# --- Grafi: hapat e vendosjes se aplikacionit ---
emrat = {0: "Python", 1: "pip", 2: "bibliotekat", 3: ".env", 4: "app"}

edges_DAG = [(0, 1), (0, 2), (1, 2), (2, 4), (3, 4)]
n_DAG = 5


# --- Funksionet nga Ushtrimi 5/6 — mos i ndrysho ---

def build_predecessors(edges, n):
    """Nderton fjalor pred ku pred[v] = lista e paraardhesve te v.
    Identik me build_predecessors nga Ushtrimi 5/6."""
    pred = {i: [] for i in range(n)}
    for u, v in edges:
        pred[v].append(u)
    return pred


def available_nodes(pred, placed, n):
    """Kthen kulmet e gatshme: te pa vendosura me te gjithe paraardhesit ne placed.
    Identik me available_nodes nga Ushtrimi 5/6."""
    return [v for v in range(n)
            if v not in placed
            and all(u in placed for u in pred[v])]


# --- Detyre: ploteso gjenero_renditjen ---

def gjenero_renditjen(edges, n):
    """
    Gjeneron nje renditje topologjike te vlefshme duke perdorur
    build_predecessors dhe available_nodes nga laboratori.
    Kthen listen boshe nese grafi ka cikel.
    """
    pred   = build_predecessors(___, ___)    # PLOTESO: thirr build_predecessors(edges, n)
    placed = set()
    rendi  = []

    while len(rendi) < n:
        avail = available_nodes(___, ___, ___)  # PLOTESO: thirr available_nodes(pred, placed, n)

        if not avail:                           # asnje kulm i disponueshem -> cikel
            print("Grafi ka cikel — renditja nuk ekziston!")
            return []

        v = avail[0]                            # zgjedh kulmin e pare te disponueshem
        placed.add(v)
        rendi.append(v)

    return rendi


# --- Testimi ---
rendi = gjenero_renditjen(edges_DAG, n_DAG)

print("Renditja topologjike e gjetur:")
for pozita, v in enumerate(rendi, 1):
    print(f"  {pozita}. [{v}] {emrat[v]}")

# Verifikimi: per cdo brinje (u,v), u duhet te jete para v ne rend
pred_test = build_predecessors(edges_DAG, n_DAG)
pozicion  = {v: i for i, v in enumerate(rendi)}
e_sakte   = all(pozicion[u] < pozicion[v]
                for v in range(n_DAG)
                for u in pred_test[v])
print(f"\nRenditja {'E SAKTE [OK]' if e_sakte else 'E GABUAR [ERR]'}")


# --- Testi me cikel ---
edges_CIKEL = [(0, 1), (1, 2), (2, 0)]   # 0->1->2->0 cikel!
print("\n--- Test me cikel ---")
gjenero_renditjen(edges_CIKEL, 3)